# 05. Blocagem

Quando você conhece de antemão uma variável categórica que afeta muito o resultado (região, tipo de dispositivo, safra), dá para **garantir** equilíbrio nela randomizando **dentro** de cada grupo. Isso é blocagem: `BlockedCRD` para o desenho e `BlockedDifferenceInMeans` para o estimador.

## O experimento

Dois blocos, `A` e `B`, com linhas de base bem diferentes. O efeito verdadeiro é `0.5` em ambos. A blocagem randomiza dentro de cada bloco, então a proporção tratado/controle é preservada em cada um.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import BlockedAssignment
from skxperiments.design.blocked_crd import BlockedCRD

rng = np.random.default_rng(5)
n_per = 50
blocks = np.repeat(["A", "B"], n_per)
n = len(blocks)

block_baseline = np.where(blocks == "A", 0.0, 3.0)   # bloco B parte mais alto
tau = 0.5
y0 = block_baseline + rng.normal(size=n)
y1 = y0 + tau

df = pd.DataFrame({"block": blocks})
design = BlockedCRD(block_col="block", p=0.5, seed=5)
assignment = design.randomize(df)
t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
assignment = BlockedAssignment(
    data=data,
    treatment_col=assignment.treatment_col_,
    design=design,
    block_col=assignment.block_col_,
    block_sizes=assignment.block_sizes_,
    seed=5,
)

## Estimar com média ponderada por bloco

`BlockedDifferenceInMeans` estima o efeito dentro de cada bloco e combina pela proporção de unidades, o que é não enviesado sob blocagem.

In [ ]:
from skxperiments.estimators.blocked_difference_in_means import (
    BlockedDifferenceInMeans,
)

est = BlockedDifferenceInMeans(outcome_col="y").fit(assignment)
result = est.estimate()
print(f"ATE (blocado): {result.ate:.3f}  | verdadeiro: {tau}")
print("ATE por bloco:", {k: round(v, 3) for k, v in est.block_ates_.items()})

## O que aprendemos

- Blocagem **garante** equilíbrio na variável de bloco, em vez de deixar ao acaso.
- O estimador combina os efeitos por bloco ponderando pelo tamanho, e isola a grande diferença de base entre `A` e `B`.
- É a escolha natural quando uma covariável categórica conhecida domina a variação do resultado.

**Próximo:** `06. Fatorial` estima efeitos principais e interações de vários fatores.